# Phase 5 — Legal Statute Identification (LSI): Experiment 2 = Phase 1 Encoder + Partial (Layer-wise) Fine-Tuning

**Technique.** Partial fine-tuning: the Phase 1 bidirectional-GQA encoder starts from its pretrained weights, the **embedding table and the lower `N_FROZEN` transformer blocks are frozen**, and only the **top `UNFREEZE_TOP_N` blocks + final norm + the new classification head** are trained. Frozen layers run under `no_grad`, so there is no backward pass through them (less memory, faster epochs).

**No experiment sweep here.** The recipe below is the best one found in Experiment 1 (full fine-tuning), reused as-is:

| Choice | Value (from Exp 1) |
|---|---|
| Long-document representation | `head` (first 510 tok) + `cue_rest` (cue-selected sentences from the rest), 2 views, joint token pooling |
| Pooling / head | learned attention pooling (`attn`) |
| Loss | BCE with per-statute positive weight `sqrt(neg/pos)` clipped to [1,20] |
| Encoder LR / head LR / LLRD | 1e-4 / 1e-3 / 0.92, warm-up 6 %, linear decay, AMP fp16 |
| Post-processing | chosen out-of-fold on dev from: global / per-label-shrunk thresholds, top-1 fallback, temperature, Platt, co-occurrence stacker |

**Final run:** full train set, **max 6 epochs, early stopping patience 2** on dev macro-F1, saves `.pt` + `submission.json`.

## 0. Configuration (edit this cell only)

In [1]:
import os, sys, re, gc, json, math, time, heapq, random, subprocess, warnings
from pathlib import Path
from dataclasses import dataclass, fields, asdict
from collections import OrderedDict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

ON_KAGGLE = Path("/kaggle").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("./lsi_work")
WORK.mkdir(parents=True, exist_ok=True)

PHASE1_CHECKPOINT_OVERRIDE = "/kaggle/input/datasets/vishwaspathania/encoder-phase1/best_encoder_mlm.pt"
PHASE1_TOKENIZER_OVERRIDE  = "/kaggle/input/datasets/vishwaspathania/tokenizer-phase5/legal_bpe_a1_50k_final.json"

LOCAL_DATA_DIR = ""          # folder with train.json/dev.json/test.json, or "" to download from Hugging Face
HF_REPO_ID = "Exploration-Lab/CS-781-Capstone"
HF_FILES = {"train": "Phase_5_LSI/Train/train.json", "dev": "Phase_5_LSI/Train/dev.json", "test": "Phase_5_LSI/Test/test.json"}

NUM_LABELS = 100
MAX_LEN = 512
CONTENT = MAX_LEN - 2
MAX_DOC_TOKENS = 16_384
TOK_CACHE_VERSION = "v1"      # same version as Exp 1 -> the token cache is reused if it is still in /kaggle/working

# ---------------- partial fine-tuning setting ----------------
N_LAYERS_TOTAL = 16
UNFREEZE_TOP_N = 6            # train only the top 6 of 16 blocks (+ final norm + head); embeddings + lower 10 blocks frozen
N_FROZEN = N_LAYERS_TOTAL - UNFREEZE_TOP_N

# ---------------- best recipe found in Experiment 1 (no re-search) ----------------
RECIPE = dict(views=("head", "cue_rest"), view_agg="tokens", head="attn", loss="bce_posw",
              enc_lr=1e-4, head_lr=1e-3, llrd=0.92, wd=0.01, dropout=0.1, warmup=0.06)
EFFECTIVE_BATCH = 32

# ---------------- final training ----------------
FINAL_MAX_EPOCHS = 6
FINAL_PATIENCE   = 2
RESUME_FINAL     = False      # True -> continue an interrupted run from final_last.pt
SMOKE_TEST       = False      # True -> tiny dry run of the whole pipeline
FINAL_TRAIN_DOCS_LIMIT = None
EXP1_DEV_MACRO_F1 = 0.3826    # Exp 1 (full FT) best-epoch dev macro-F1 @ global threshold, for the comparison printout
SEED = 42

if SMOKE_TEST: FINAL_MAX_EPOCHS, FINAL_TRAIN_DOCS_LIMIT = 2, 512
RUN_DIR = WORK / ("lsi_exp2_partial_smoke" if SMOKE_TEST else "lsi_exp2_partial"); RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = WORK / f"lsi_tok_cache_{TOK_CACHE_VERSION}"; CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("WORK:", WORK.resolve(), "| ON_KAGGLE:", ON_KAGGLE)
print(f"partial tuning: frozen = embeddings + blocks 0..{N_FROZEN-1}; trainable = blocks {N_FROZEN}..{N_LAYERS_TOTAL-1} + final_norm + head")

WORK: /kaggle/working | ON_KAGGLE: True
partial tuning: frozen = embeddings + blocks 0..9; trainable = blocks 10..15 + final_norm + head


## 1. Imports, GPU, seeds

In [2]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
try:
    from IPython.display import display
except Exception:
    display = print

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_everything(SEED)
assert torch.cuda.is_available(), "Enable a GPU runtime."
device = torch.device("cuda")
gpu = torch.cuda.get_device_properties(0)
print("GPU:", gpu.name, f"| {gpu.total_memory/1024**3:.1f} GiB | torch {torch.__version__}")

GPU: Tesla T4 | 14.6 GiB | torch 2.11.0+cu128


## 2. Locate / download data and Phase 1 artifacts

In [3]:
try:
    import ijson
except ImportError:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ijson"]); import ijson
    except Exception:
        ijson = None

def _get_hf_token():
    tok = os.environ.get("HF_TOKEN", "").strip()
    if not tok:
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            tok = ""
    if not tok:
        from getpass import getpass
        tok = getpass("Hugging Face token: ")
    return tok

def locate_data():
    paths = {}
    if LOCAL_DATA_DIR:
        for split in HF_FILES:
            p = Path(LOCAL_DATA_DIR) / f"{split}.json"
            if not p.is_file(): raise FileNotFoundError(p)
            paths[split] = p
        return paths
    from huggingface_hub import hf_hub_download
    token = _get_hf_token()
    for split, fn in HF_FILES.items():
        paths[split] = Path(hf_hub_download(repo_id=HF_REPO_ID, repo_type="dataset", filename=fn, token=token))
    return paths

def iter_json_records(path):
    with open(path, "rb") as f:
        first = b""
        while True:
            ch = f.read(1)
            if not ch: break
            if ch not in b" \t\r\n": first = ch; break
        f.seek(0)
        if first == b"[":
            if ijson is not None: yield from ijson.items(f, "item")
            else: yield from json.load(f)
        elif first == b"{":
            for line in f:
                line = line.strip()
                if line: yield json.loads(line)
        else:
            raise ValueError(f"Unrecognised JSON layout: {path}")

def find_artifact(filename, override=""):
    if override and Path(override).is_file(): return Path(override).resolve()
    hits = set()
    for root in [Path("/kaggle/input"), WORK, Path(".")]:
        if root.exists(): hits.update(p.resolve() for p in root.rglob(filename) if p.is_file())
    hits = sorted(hits)
    if len(hits) != 1: raise FileNotFoundError(f"{filename}: found {len(hits)} candidates {hits}. Set the override path in the config cell.")
    return hits[0]

need_raw = not all((CACHE_DIR / f"{s}_done.flag").exists() for s in ["train", "dev", "test"])
paths = locate_data() if need_raw else {}
PHASE1_CHECKPOINT_PATH = find_artifact("best_encoder_mlm.pt", PHASE1_CHECKPOINT_OVERRIDE)
PHASE1_TOKENIZER_PATH  = find_artifact("legal_bpe_a1_50k_final.json", PHASE1_TOKENIZER_OVERRIDE)
print("data:", {k: str(v) for k, v in paths.items()} if paths else "(token cache present, raw files not needed)")
print("checkpoint:", PHASE1_CHECKPOINT_PATH); print("tokenizer :", PHASE1_TOKENIZER_PATH)

Hugging Face token:  ········


Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

data: {'train': '/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_5_LSI/Train/train.json', 'dev': '/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_5_LSI/Train/dev.json', 'test': '/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_5_LSI/Test/test.json'}
checkpoint: /kaggle/input/datasets/vishwaspathania/encoder-phase1/best_encoder_mlm.pt
tokenizer : /kaggle/input/datasets/vishwaspathania/tokenizer-phase5/legal_bpe_a1_50k_final.json


## 3. Phase 1 tokenizer and encoder definition (with partial-freezing support)

In [4]:
SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]
N_SPECIAL, N_BYTES = 4, 256
BASE_VOCAB = N_SPECIAL + N_BYTES
WORD_RE = re.compile(r"\S+")

class A1BPETokenizer:
    def __init__(self, itos, merges):
        self.itos = list(itos); self.merges = [tuple(m) for m in merges]
        self.vocab_size = len(self.itos); self.stoi = {s: i for i, s in enumerate(self.itos)}
        self._byte_to_id = {bytes([b]): N_SPECIAL + b for b in range(256)}
        self._merge_ranks = {pair: r for r, pair in enumerate(self.merges)}
    def _encode_word(self, word):
        raw = word.encode("utf-8"); seq = [self._byte_to_id[bytes([b])] for b in raw]
        if len(seq) < 2 or not self._merge_ranks: return seq
        n = len(seq); prev = [-1] + list(range(n - 1)); nxt = list(range(1, n)) + [-1]
        active = [True] * n; heap = []
        for i in range(n - 1):
            r = self._merge_ranks.get((seq[i], seq[i + 1]))
            if r is not None: heapq.heappush(heap, (r, i, i + 1))
        while heap:
            rank, left, right = heapq.heappop(heap)
            if not active[left] or not active[right]: continue
            if nxt[left] != right or prev[right] != left: continue
            if self._merge_ranks.get((seq[left], seq[right])) != rank: continue
            seq[left] = BASE_VOCAB + rank; active[right] = False
            rn = nxt[right]; nxt[left] = rn
            if rn != -1: prev[rn] = left
            lp = prev[left]
            if lp != -1:
                nr = self._merge_ranks.get((seq[lp], seq[left]))
                if nr is not None: heapq.heappush(heap, (nr, lp, left))
            if rn != -1:
                nr = self._merge_ranks.get((seq[left], seq[rn]))
                if nr is not None: heapq.heappush(heap, (nr, left, rn))
        out, i = [], 0
        while i != -1:
            if active[i]: out.append(seq[i])
            i = nxt[i]
        return out

def load_a1_tokenizer(path):
    with open(path, "r", encoding="utf-8") as f: payload = json.load(f)
    tok = A1BPETokenizer(payload["itos"], payload["merges"])
    assert tok.vocab_size == 50_000 and tok.itos[:4] == SPECIAL_TOKENS
    return tok

phase1_tokenizer = load_a1_tokenizer(PHASE1_TOKENIZER_PATH)
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3
SPACE_ID = N_SPECIAL + 32

_ckpt = torch.load(PHASE1_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
PHASE1_STATE = {k: v for k, v in _ckpt["model_state_dict"].items()}
_cfg_dict = dict(_ckpt["model_config"]); del _ckpt; gc.collect()

@dataclass
class ModelConfig:
    vocab_size: int = 50_000; d_model: int = 768; n_layers: int = 16; n_heads: int = 12
    n_kv_heads: int = 4; d_ff: int = 2048; max_seq_len: int = 512
    rope_base: float = 10_000.0; rms_norm_eps: float = 1e-6; dropout: float = 0.0

cfg = ModelConfig(**{k: v for k, v in _cfg_dict.items() if k in {f.name for f in fields(ModelConfig)}})
assert cfg.n_layers == N_LAYERS_TOTAL, f"set N_LAYERS_TOTAL={cfg.n_layers}"
print(cfg)

def build_rope_cache(max_seq_len, d_head, base=10_000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head // 2, dtype=torch.float32) * 2 / d_head))
    freqs = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv_freq)
    ang = torch.cat([freqs, freqs], dim=-1)
    return ang.cos(), ang.sin()
def rotate_half(x):
    x1, x2 = torch.chunk(x, 2, dim=-1); return torch.cat([-x2, x1], dim=-1)
def apply_rope(x, cos, sin):
    return x * cos.to(x.device, x.dtype)[None, None] + rotate_half(x) * sin.to(x.device, x.dtype)[None, None]

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        dt = x.dtype; xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight.float()).to(dt)

class SwiGLU(nn.Module):
    def __init__(self, d, ff):
        super().__init__()
        self.gate_proj = nn.Linear(d, ff, bias=False); self.up_proj = nn.Linear(d, ff, bias=False); self.down_proj = nn.Linear(ff, d, bias=False)
    def forward(self, x): return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class BidirectionalGQAAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.d_model, self.n_heads, self.n_kv_heads = c.d_model, c.n_heads, c.n_kv_heads
        self.head_dim = c.d_model // c.n_heads; self.n_rep = c.n_heads // c.n_kv_heads; self.max_seq_len = c.max_seq_len
        self.q_proj = nn.Linear(c.d_model, c.n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(c.d_model, c.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(c.d_model, c.n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(c.d_model, c.d_model, bias=False)
        cos, sin = build_rope_cache(c.max_seq_len, self.head_dim, c.rope_base)
        self.register_buffer("rope_cos", cos, persistent=False); self.register_buffer("rope_sin", sin, persistent=False)
    def _split(self, x, n):
        b, t, _ = x.shape; return x.reshape(b, t, n, self.head_dim).transpose(1, 2)
    def forward(self, x, key_mask=None):
        b, t, _ = x.shape
        if t > self.max_seq_len: raise ValueError(f"sequence {t} > {self.max_seq_len}")
        q = apply_rope(self._split(self.q_proj(x), self.n_heads), self.rope_cos[:t], self.rope_sin[:t])
        k = apply_rope(self._split(self.k_proj(x), self.n_kv_heads), self.rope_cos[:t], self.rope_sin[:t])
        v = self._split(self.v_proj(x), self.n_kv_heads)
        k = k.repeat_interleave(self.n_rep, dim=1); v = v.repeat_interleave(self.n_rep, dim=1)
        mask = None if key_mask is None else key_mask[:, None, None, :]
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0.0, is_causal=False)
        return self.out_proj(a.transpose(1, 2).contiguous().view(b, t, self.d_model))

class EncoderBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.attn_norm = RMSNorm(c.d_model, c.rms_norm_eps); self.attn = BidirectionalGQAAttention(c)
        self.ffn_norm = RMSNorm(c.d_model, c.rms_norm_eps); self.mlp = SwiGLU(c.d_model, c.d_ff)
    def forward(self, x, key_mask=None):
        x = x + self.attn(self.attn_norm(x), key_mask)
        return x + self.mlp(self.ffn_norm(x))

class LegalEncoder(nn.Module):
    """Phase 1 encoder. The embedding table and the first `n_frozen` blocks are frozen and executed under no_grad."""
    def __init__(self, c, grad_ckpt=True, n_frozen=0):
        super().__init__(); self.config = c; self.grad_ckpt = grad_ckpt; self.n_frozen = n_frozen
        self.token_embedding = nn.Embedding(c.vocab_size, c.d_model)
        self.layers = nn.ModuleList([EncoderBlock(c) for _ in range(c.n_layers)])
        self.final_norm = RMSNorm(c.d_model, c.rms_norm_eps)
    def freeze_lower(self):
        if self.n_frozen <= 0: return
        for p in self.token_embedding.parameters(): p.requires_grad_(False)
        for layer in self.layers[:self.n_frozen]:
            for p in layer.parameters(): p.requires_grad_(False)
    def forward(self, input_ids, key_mask=None):
        if self.n_frozen > 0:
            with torch.no_grad():
                x = self.token_embedding(input_ids)
                for layer in self.layers[:self.n_frozen]: x = layer(x, key_mask)
        else:
            x = self.token_embedding(input_ids)
        for layer in self.layers[self.n_frozen:]:
            if self.grad_ckpt and self.training and torch.is_grad_enabled():
                x = checkpoint(layer, x, key_mask, use_reentrant=False)
            else:
                x = layer(x, key_mask)
        return self.final_norm(x)

def load_phase1_encoder_weights(enc, state):
    cleaned = {}
    for k, v in state.items():
        for pre in ("module.", "_orig_mod."):
            if k.startswith(pre): k = k[len(pre):]
        cleaned[k] = v
    cleaned.pop("mlm_head.weight", None)
    enc.load_state_dict(cleaned, strict=True); return enc

_probe = LegalEncoder(cfg, grad_ckpt=False); load_phase1_encoder_weights(_probe, PHASE1_STATE)
ENCODER_PARAMS = sum(p.numel() for p in _probe.parameters()); del _probe
print(f"Phase 1 encoder verified: {ENCODER_PARAMS:,} parameters")

ModelConfig(vocab_size=50000, d_model=768, n_layers=16, n_heads=12, n_kv_heads=4, d_ff=2048, max_seq_len=512, rope_base=10000.0, rms_norm_eps=1e-06, dropout=0.0)
Phase 1 encoder verified: 139,088,640 parameters


## 4. Token cache (full documents, tokenised once; reused from Exp 1 if present)

In [5]:
CUE_RE = re.compile(r"<(?:SECTION|ACT)>|^(?:[Ss]ections?|[Ss]ec\.?|[Aa]rticles?|[Rr]ules?|[Aa]ct|[Aa]cts|IPC|CrPC|Cr\.P\.C\.?|Code)\W*$")
SENT_END_CHARS = (".", ";", "?", "!", ":")
FLAG_CUE, FLAG_SENT = 1, 2

class FastDocTokenizer:
    def __init__(self, tok, max_doc_tokens=MAX_DOC_TOKENS, tail_keep=4096, cache_limit=3_000_000):
        self.tok, self.max_doc_tokens, self.tail_keep, self.cache_limit = tok, max_doc_tokens, tail_keep, cache_limit; self.cache = {}
    def _word(self, w):
        r = self.cache.get(w)
        if r is None:
            ids = self.tok._encode_word(w); fl = [0] * len(ids)
            if CUE_RE.search(w): fl[0] |= FLAG_CUE
            if w.endswith(SENT_END_CHARS): fl[-1] |= FLAG_SENT
            r = (tuple(ids), tuple(fl))
            if len(self.cache) > self.cache_limit: self.cache.clear()
            self.cache[w] = r
        return r
    def encode(self, text):
        ids, fl, first = [], [], True
        for m in WORD_RE.finditer(text):
            if not first: ids.append(SPACE_ID); fl.append(0)
            w_ids, w_fl = self._word(m.group(0)); ids.extend(w_ids); fl.extend(w_fl); first = False
        n = len(ids)
        if n > self.max_doc_tokens:
            cut = self.max_doc_tokens - self.tail_keep
            ids = ids[:cut] + ids[-self.tail_keep:]; fl = fl[:cut] + fl[-self.tail_keep:]
        return np.asarray(ids, dtype=np.uint16), np.asarray(fl, dtype=np.uint8), n

_WORKER = {}
def _init_worker(): _WORKER["t"] = FastDocTokenizer(phase1_tokenizer)
def _tok_batch(texts):
    t = _WORKER["t"]; out = []
    for s in texts:
        i, f, n = t.encode(s); out.append((i, f, n))
    return out

def build_split_cache(split):
    flag = CACHE_DIR / f"{split}_done.flag"
    if flag.exists(): return
    has_labels = split != "test"; ids_raw, texts, labels = [], [], []
    for row in tqdm(iter_json_records(paths[split]), desc=f"read {split}"):
        ids_raw.append(row["id"]); texts.append(row["text"])
        if has_labels:
            lab = sorted(set(int(x) for x in row["labels"])); assert all(0 <= x < NUM_LABELS for x in lab); labels.append(lab)
    n_docs = len(texts); batches = [texts[i:i + 128] for i in range(0, n_docs, 128)]; results = []
    try:
        import multiprocessing as mp
        ctx = mp.get_context("fork"); nproc = min(4, os.cpu_count() or 1)
        with ctx.Pool(nproc, initializer=_init_worker) as pool:
            for r in tqdm(pool.imap(_tok_batch, batches), total=len(batches), desc=f"tokenise {split}"): results.extend(r)
    except Exception as e:
        print("parallel tokenisation unavailable (", type(e).__name__, ") -> sequential"); _init_worker(); results = []
        for b in tqdm(batches, desc=f"tokenise {split} (seq)"): results.extend(_tok_batch(b))
    lengths = np.array([len(r[0]) for r in results], dtype=np.int64)
    offsets = np.zeros(n_docs + 1, dtype=np.int64); offsets[1:] = np.cumsum(lengths)
    np.save(CACHE_DIR / f"{split}_tokens.npy", np.concatenate([r[0] for r in results]))
    np.save(CACHE_DIR / f"{split}_flags.npy", np.concatenate([r[1] for r in results]))
    np.save(CACHE_DIR / f"{split}_offsets.npy", offsets)
    np.save(CACHE_DIR / f"{split}_orig_len.npy", np.array([r[2] for r in results], dtype=np.int64))
    if has_labels:
        Y = np.zeros((n_docs, NUM_LABELS), dtype=np.uint8)
        for i, lab in enumerate(labels): Y[i, lab] = 1
        np.save(CACHE_DIR / f"{split}_labels.npy", Y)
    with open(CACHE_DIR / f"{split}_case_ids.json", "w", encoding="utf-8") as f: json.dump(ids_raw, f)
    flag.write_text("ok"); del results, texts; gc.collect()
    print(f"{split}: {n_docs:,} docs | {lengths.sum():,} tokens")

t0 = time.time()
for s in ["train", "dev", "test"]: build_split_cache(s)
print(f"token cache ready in {(time.time()-t0)/60:.1f} min -> {CACHE_DIR}")

def load_arrays(split, has_labels=True):
    d = dict(tokens=np.load(CACHE_DIR / f"{split}_tokens.npy", mmap_mode="r"), flags=np.load(CACHE_DIR / f"{split}_flags.npy", mmap_mode="r"),
             offsets=np.load(CACHE_DIR / f"{split}_offsets.npy"), orig_len=np.load(CACHE_DIR / f"{split}_orig_len.npy"))
    d["ids"] = json.load(open(CACHE_DIR / f"{split}_case_ids.json"))
    if has_labels: d["labels"] = np.load(CACHE_DIR / f"{split}_labels.npy")
    return d

ARR = {"train": load_arrays("train"), "dev": load_arrays("dev"), "test": load_arrays("test", has_labels=False)}
Ytr, Ydv = ARR["train"]["labels"], ARR["dev"]["labels"]
PRIOR = np.clip(Ytr.mean(0), 1e-5, 1 - 1e-5); POS_COUNT = Ytr.sum(0).astype(np.float64)
PRIOR_LOGIT = np.log(PRIOR / (1 - PRIOR)).astype(np.float32)
Ntr, Ndv, Nte = len(Ytr), len(Ydv), len(ARR["test"]["ids"])
print(f"train {Ntr:,} | dev {Ndv:,} | test {Nte:,} | labels/doc {Ytr.sum(1).mean():.2f}")

read train: 0it [00:00, ?it/s]

tokenise train:   0%|          | 0/334 [00:00<?, ?it/s]

train: 42,750 docs | 102,823,711 tokens


read dev: 0it [00:00, ?it/s]

tokenise dev:   0%|          | 0/80 [00:00<?, ?it/s]

dev: 10,181 docs | 27,323,268 tokens


read test: 0it [00:00, ?it/s]

tokenise test:   0%|          | 0/102 [00:00<?, ?it/s]

test: 13,019 docs | 33,541,917 tokens
token cache ready in 0.9 min -> /kaggle/working/lsi_tok_cache_v1
train 42,750 | dev 10,181 | test 13,019 | labels/doc 3.70


## 5. Document views: `head` + `cue_rest` (best representation from Exp 1)

In [6]:
MAX_SEG, CUE_CAP, LAST_BONUS_TOKENS = 96, 3, 128

def select_cue_tokens(tok, fl, budget=CONTENT, head_keep=128):
    n = len(tok)
    if n <= budget: return tok
    head = tok[:head_keep]; rest = tok[head_keep:]; rfl = fl[head_keep:]; budget_rest = budget - len(head); nr = len(rest)
    ends = np.flatnonzero(rfl & FLAG_SENT) + 1; bounds = [0] + ends.tolist()
    if bounds[-1] != nr: bounds.append(nr)
    starts_l, ends_l = [], []
    for a, b in zip(bounds[:-1], bounds[1:]):
        while b - a > MAX_SEG: starts_l.append(a); ends_l.append(a + MAX_SEG); a += MAX_SEG
        if b > a: starts_l.append(a); ends_l.append(b)
    starts = np.asarray(starts_l); ends_a = np.asarray(ends_l)
    cs = np.concatenate([[0], np.cumsum((rfl & FLAG_CUE).astype(np.int64))])
    score = np.minimum(cs[ends_a] - cs[starts], CUE_CAP).astype(np.float64)
    score += 0.5 * (starts >= nr - LAST_BONUS_TOKENS); score -= np.arange(len(starts)) * 1e-6
    order = np.argsort(-score, kind="stable"); chosen, used = [], 0
    for k in order:
        s0, e0 = int(starts[k]), int(ends_a[k]); ln = e0 - s0; room = budget_rest - used
        if ln <= room: chosen.append((s0, e0)); used += ln
        elif room >= 16: chosen.append((s0, s0 + room)); used += room
        if budget_rest - used < 8: break
    chosen.sort(); pieces = [rest[s0:e0] for s0, e0 in chosen]
    return np.concatenate([head] + pieces) if pieces else head

def make_view(name, tok, fl):
    n = len(tok)
    if name == "head": return tok[:CONTENT]
    if name == "cue_rest":
        if n <= CONTENT: return tok[:0]
        return select_cue_tokens(tok[CONTENT:], fl[CONTENT:], CONTENT, head_keep=0)
    raise ValueError(name)

class LSIDataset(Dataset):
    def __init__(self, split, indices, views, with_labels=True):
        self.a = ARR[split]; self.indices = np.asarray(indices, dtype=np.int64); self.views = tuple(views)
        self.with_labels = with_labels; self.need_flags = any(v.startswith("cue") for v in views)
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        d = int(self.indices[i]); a = self.a; o0, o1 = int(a["offsets"][d]), int(a["offsets"][d + 1])
        tok = np.asarray(a["tokens"][o0:o1]); fl = np.asarray(a["flags"][o0:o1]) if self.need_flags else None
        seqs, valid = [], []
        for v in self.views:
            c = make_view(v, tok, fl); seqs.append(np.concatenate([[BOS_ID], c.astype(np.int64), [EOS_ID]])); valid.append(len(c) > 0)
        valid[0] = True
        item = {"seqs": seqs, "valid": valid, "idx": d}
        if self.with_labels: item["y"] = a["labels"][d].astype(np.float32)
        return item

def collate_lsi(batch):
    B, V = len(batch), len(batch[0]["seqs"]); T = max(len(s) for b in batch for s in b["seqs"])
    ids = torch.full((B, V, T), PAD_ID, dtype=torch.long)
    for i, b in enumerate(batch):
        for v, s in enumerate(b["seqs"]): ids[i, v, :len(s)] = torch.from_numpy(s)
    out = {"input_ids": ids, "mask": ids.ne(PAD_ID), "valid": torch.tensor([b["valid"] for b in batch]), "idx": torch.tensor([b["idx"] for b in batch])}
    if "y" in batch[0]: out["y"] = torch.from_numpy(np.stack([b["y"] for b in batch]))
    return out

_demo = LSIDataset("train", [0], RECIPE["views"])[0]
print("example item views:", [len(s) for s in _demo["seqs"]], "valid", _demo["valid"])

example item views: [512, 512] valid [True, True]


## 6. Evaluation + post-processing toolkit (same as Exp 1)

In [7]:
from scipy.ndimage import uniform_filter1d
from sklearn.metrics import average_precision_score

GRID = np.round(np.arange(0.01, 0.96, 0.01), 2)
sigmoid = lambda z: 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def f1_grid(y, p, grid=GRID):
    y = y.astype(bool); npos = y.sum(0); G, C = len(grid), y.shape[1]
    tp = np.empty((G, C)); fp = np.empty((G, C))
    for gi, g in enumerate(grid):
        pr = p >= g; tp[gi] = (pr & y).sum(0); fp[gi] = pr.sum(0) - tp[gi]
    fn = npos[None, :] - tp
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)

def best_global(y, p, grid=GRID):
    f1 = f1_grid(y, p, grid); macro = f1.mean(1); i = int(np.argmax(macro)); return float(grid[i]), float(macro[i])

def fit_per_label(y, p, t_glob, min_pos=10, shrink=100.0, grid=GRID):
    f1 = uniform_filter1d(f1_grid(y, p, grid), size=3, axis=0, mode="nearest")
    thr = grid[np.argmax(f1, axis=0)].astype(np.float64); sup = y.sum(0).astype(np.float64)
    thr = np.where(sup < min_pos, t_glob, thr)
    w = sup / (sup + shrink) if shrink > 0 else np.ones_like(sup)
    return w * thr + (1 - w) * t_glob

def predict_binary(p, thr, ensure_one=True):
    pred = p >= thr
    if ensure_one:
        empty = ~pred.any(1)
        if empty.any(): pred[empty, np.argmax(p[empty] / thr[None, :] if np.ndim(thr) else p[empty], axis=1)] = True
    return pred.astype(np.uint8)

def label_f1(y, pred):
    y = y.astype(bool); pr = pred.astype(bool); tp = (y & pr).sum(0); fp = (~y & pr).sum(0); fn = (y & ~pr).sum(0)
    return 2 * tp / np.maximum(2 * tp + fp + fn, 1)

def prf_summary(y, pred):
    yb, pb = y.astype(bool), pred.astype(bool)
    tp = (yb & pb).sum(0); fp = (~yb & pb).sum(0); fn = (yb & ~pb).sum(0)
    prec = tp / np.maximum(tp + fp, 1); rec = tp / np.maximum(tp + fn, 1)
    return dict(macro_f1=float(label_f1(y, pred).mean()), micro_f1=float(2 * tp.sum() / max(2 * tp.sum() + fp.sum() + fn.sum(), 1)),
                macro_precision=float(prec.mean()), macro_recall=float(rec.mean()), subset_acc=float((yb == pb).all(1).mean()),
                avg_pred=float(pb.sum(1).mean()), empty_rate=float((pb.sum(1) == 0).mean()))

def fit_temperature(y, z):
    best, bt = 1e9, 1.0
    for T in np.arange(0.5, 4.01, 0.1):
        p = np.clip(sigmoid(z / T), 1e-7, 1 - 1e-7); l = -(y * np.log(p) + (1 - y) * np.log(1 - p)).mean()
        if l < best: best, bt = l, T
    def tf(z2, T=bt): return z2 / T
    tf.params = {"type": "temperature", "T": float(bt)}; return tf

def fit_platt(y, z, lam=1.0, iters=30):
    C = y.shape[1]; a = np.ones(C); b = np.zeros(C)
    for _ in range(iters):
        p = sigmoid(z * a + b); r = p - y; w = p * (1 - p)
        ga = (r * z).sum(0) + lam * (a - 1); gb = r.sum(0) + lam * b
        haa = (w * z * z).sum(0) + lam; hab = (w * z).sum(0); hbb = w.sum(0) + lam
        det = haa * hbb - hab ** 2 + 1e-12
        a -= (hbb * ga - hab * gb) / det; b -= (-hab * ga + haa * gb) / det; a = np.clip(a, 0.05, 5.0)
    def tf(z2, a=a, b=b): return z2 * a + b
    tf.params = {"type": "platt", "a": a.tolist(), "b": b.tolist()}; return tf

def fit_stacker(y, z, l2=0.01, steps=100, lr=0.02, seed=0):
    N, C = z.shape; a = np.ones(C); b = np.zeros(C); W = np.zeros((C, C)); eye = 1 - np.eye(C)
    params = [a, b, W]; m = [np.zeros_like(q) for q in params]; v = [np.zeros_like(q) for q in params]; zc = np.clip(z, -12, 12)
    for t in range(1, steps + 1):
        logit = zc * a + zc @ W + b; g = (sigmoid(logit) - y) / N
        ga = (g * zc).sum(0); gb = g.sum(0); gW = (zc.T @ g) * eye + l2 * W
        for k, gr in enumerate([ga, gb, gW]):
            m[k] = 0.9 * m[k] + 0.1 * gr; v[k] = 0.999 * v[k] + 0.001 * gr ** 2
            params[k] -= lr * (m[k] / (1 - 0.9 ** t)) / (np.sqrt(v[k] / (1 - 0.999 ** t)) + 1e-8)
        W *= eye
    def tf(z2, a=a, b=b, W=W): return np.clip(z2, -12, 12) * a + np.clip(z2, -12, 12) @ W + b
    tf.params = {"type": "stacker", "a": a.tolist(), "b": b.tolist(), "W": W.tolist()}; return tf

def stack_oof(y, z, l2, K=3):
    idx = np.random.default_rng(1).permutation(len(z)); folds = np.array_split(idx, K); out = np.zeros_like(z)
    for k in range(K):
        te = folds[k]; tr = np.concatenate([folds[j] for j in range(K) if j != k]); out[te] = fit_stacker(y[tr], z[tr], l2)(z[te])
    return out

def make_strategy(calib=None, thr="global", shrink=100.0, min_pos=10, top1=True, l2=0.01):
    def fit(y, z):
        if calib == "temp": tf = fit_temperature(y, z); z_thr = tf(z)
        elif calib == "platt": tf = fit_platt(y, z); z_thr = tf(z)
        elif calib == "stack": tf = fit_stacker(y, z, l2); z_thr = stack_oof(y, z, l2)
        else:
            def tf(q): return q
            tf.params = {"type": "identity"}; z_thr = z
        p_tr = sigmoid(z_thr); tg, _ = best_global(y, p_tr)
        thr_vec = np.full(y.shape[1], tg) if thr == "global" else fit_per_label(y, p_tr, tg, min_pos, shrink)
        def predict(z2): return predict_binary(sigmoid(tf(z2)), thr_vec, top1)
        predict.thr = thr_vec; predict.tf = tf; return predict
    return fit

STRATEGIES = OrderedDict([
    ("global", make_strategy(None, "global", top1=False)), ("global + top1", make_strategy(None, "global")),
    ("per-label raw + top1", make_strategy(None, "label", shrink=0)), ("per-label shrunk(100) + top1", make_strategy(None, "label", shrink=100)),
    ("per-label shrunk(300) + top1", make_strategy(None, "label", shrink=300)), ("temperature -> global", make_strategy("temp", "global")),
    ("Platt -> global", make_strategy("platt", "global")), ("Platt -> per-label shrunk(100)", make_strategy("platt", "label", shrink=100)),
    ("stacker -> global", make_strategy("stack", "global")), ("stacker -> per-label shrunk(100)", make_strategy("stack", "label", shrink=100)),
])

def crossfit(y, z, fit_fn, K=5, seed=0):
    rng = np.random.default_rng(seed); folds = np.array_split(rng.permutation(len(z)), K); pred = np.zeros(y.shape, np.uint8)
    for k in range(K):
        te = folds[k]; tr = np.concatenate([folds[j] for j in range(K) if j != k]); pred[te] = fit_fn(y[tr], z[tr])(z[te])
    return pred

def ece_brier(y, p, bins=15):
    pf, yf = p.ravel(), y.ravel().astype(float); edges = np.linspace(0, 1, bins + 1); ece = 0.0
    idx = np.clip(np.digitize(pf, edges) - 1, 0, bins - 1)
    for b in range(bins):
        m = idx == b
        if m.any(): ece += m.mean() * abs(pf[m].mean() - yf[m].mean())
    return float(ece), float(((pf - yf) ** 2).mean())

def macro_ap(y, p):
    v = [average_precision_score(y[:, j], p[:, j]) for j in range(y.shape[1]) if 0 < y[:, j].sum() < len(y)]
    return float(np.mean(v)) if v else 0.0

def evaluate_logits(y, z, perlabel_cv=True):
    p = sigmoid(z); thr, mf1 = best_global(y, p); pred = (p >= thr).astype(np.uint8)
    out = prf_summary(y, pred); out["best_global_thr"] = thr; out["macro_f1_global"] = mf1
    pc = np.clip(p, 1e-7, 1 - 1e-7); out["bce"] = float(-(y * np.log(pc) + (1 - y) * np.log(1 - pc)).mean()); out["macro_ap"] = macro_ap(y, p)
    if perlabel_cv:
        pr = crossfit(y, z, STRATEGIES["per-label shrunk(100) + top1"], K=4); out["macro_f1_perlabel_cv"] = float(label_f1(y, pr).mean())
    return out

def strategy_table(y, z, K=5):
    rows, preds = [], {}
    for name, fn in STRATEGIES.items():
        t0 = time.time(); pr = crossfit(y, z, fn, K=K); preds[name] = pr
        rows.append(dict(strategy=name, **prf_summary(y, pr), seconds=time.time() - t0))
    return pd.DataFrame(rows).sort_values("macro_f1", ascending=False).reset_index(drop=True), preds
print("toolkit ready")

toolkit ready


## 7. Attention-pooling head, positive-weighted BCE, model wrapper

In [8]:
D_MODEL = cfg.d_model
def content_mask(ids, pool_mask): return pool_mask & ids.ne(BOS_ID) & ids.ne(EOS_ID)

class AttnPoolHead(nn.Module):
    """One learned query attends over all content tokens of all views (joint pooling)."""
    def __init__(self, C, dropout, d_att=256):
        super().__init__(); self.proj = nn.Linear(D_MODEL, d_att); self.v = nn.Linear(d_att, 1, bias=False)
        self.drop = nn.Dropout(dropout); self.out = nn.Linear(D_MODEL, C)
    def bias_param(self): return self.out.bias
    def forward(self, h, ids, pm):
        cm = content_mask(ids, pm)
        s = self.v(torch.tanh(self.proj(h))).squeeze(-1).float().masked_fill(~cm, -1e4)
        a = torch.softmax(s, dim=1) * cm; a = a / a.sum(1, keepdim=True).clamp_min(1e-9)
        return self.out(self.drop((a.unsqueeze(-1).to(h.dtype) * h).sum(1)))

class LSIModel(nn.Module):
    def __init__(self, encoder, dropout, prior_logit):
        super().__init__(); self.encoder = encoder; self.head = AttnPoolHead(NUM_LABELS, dropout)
        with torch.no_grad(): self.head.bias_param().copy_(torch.as_tensor(prior_logit, dtype=torch.float32))
    def forward(self, input_ids, mask, valid):
        B, V, T = input_ids.shape
        h = self.encoder(input_ids.reshape(B * V, T), key_mask=mask.reshape(B * V, T)).float()
        pm = mask & valid[:, :, None]
        return self.head(h.reshape(B, V * T, -1), input_ids.reshape(B, V * T), pm.reshape(B, V * T))

_pw = torch.as_tensor(np.clip(np.sqrt((Ntr - POS_COUNT) / POS_COUNT), 1, 20), dtype=torch.float32, device=device)
def loss_fn(z, y): return F.binary_cross_entropy_with_logits(z, y, pos_weight=_pw)
print("pos_weight range: %.2f - %.2f" % (_pw.min().item(), _pw.max().item()))

pos_weight range: 1.63 - 12.50


## 8. Training engine for partial fine-tuning
Only parameters with `requires_grad=True` reach the optimizer; layer-wise LR decay (0.92) is applied across the unfrozen blocks.

In [9]:
GRAD_CKPT, MB_SEQS, NUM_WORKERS, MAX_GRAD_NORM = False, 16, 2, 1.0
def mb_docs(): return max(1, MB_SEQS // len(RECIPE["views"]))
def accum(): return max(1, EFFECTIVE_BATCH // mb_docs())

def build_model(grad_ckpt=None):
    enc = LegalEncoder(cfg, grad_ckpt=GRAD_CKPT if grad_ckpt is None else grad_ckpt, n_frozen=N_FROZEN)
    load_phase1_encoder_weights(enc, PHASE1_STATE); enc.freeze_lower()
    return LSIModel(enc, RECIPE["dropout"], PRIOR_LOGIT).to(device)

def param_report(model):
    tot = sum(p.numel() for p in model.parameters()); tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, tr

def make_optimizer(model):
    n = cfg.n_layers; groups = {}
    for name, p in model.named_parameters():
        if not p.requires_grad: continue
        if name.startswith("encoder."):
            if "token_embedding" in name: depth = 0
            elif "layers." in name: depth = int(name.split("layers.")[1].split(".")[0]) + 1
            else: depth = n
            lr = RECIPE["enc_lr"] * (RECIPE["llrd"] ** (n - depth))
        else: lr = RECIPE["head_lr"]
        wd = 0.0 if p.ndim <= 1 else RECIPE["wd"]
        groups.setdefault((lr, wd), []).append(p)
    return torch.optim.AdamW([{"params": ps, "lr": lr, "weight_decay": wd} for (lr, wd), ps in groups.items()], betas=(0.9, 0.999))

def build_training(model, n_docs, epochs):
    mb, acc = mb_docs(), accum(); spe = math.ceil(math.ceil(n_docs / mb) / acc); total = spe * epochs; warm = max(1, int(total * RECIPE["warmup"]))
    lam = lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm))
    opt = make_optimizer(model)
    return dict(opt=opt, sched=torch.optim.lr_scheduler.LambdaLR(opt, lam), scaler=torch.amp.GradScaler("cuda"), mb=mb, accum=acc, total_steps=total)

def make_loader(ds, batch, shuffle, seed=0):
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(ds, batch_size=batch, shuffle=shuffle, generator=g if shuffle else None, collate_fn=collate_lsi, num_workers=NUM_WORKERS, pin_memory=True)

def to_dev(b): return (b["input_ids"].to(device, non_blocking=True), b["mask"].to(device, non_blocking=True), b["valid"].to(device, non_blocking=True))

def train_one_epoch(model, loader, T, tag="", log_frac=0.1):
    model.train(); opt, sched, scaler, acc = T["opt"], T["sched"], T["scaler"], T["accum"]
    trainable = [p for p in model.parameters() if p.requires_grad]
    opt.zero_grad(set_to_none=True); tot, nd, t0, n_micro = 0.0, 0, time.perf_counter(), len(loader); next_log = max(1, int(n_micro * log_frac))
    for i, b in enumerate(loader, 1):
        ids, mask, valid = to_dev(b); y = b["y"].to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16): logits = model(ids, mask, valid)
        loss = loss_fn(logits.float(), y)
        if not torch.isfinite(loss): raise RuntimeError(f"non-finite loss at micro-batch {i}")
        scaler.scale(loss / acc).backward(); tot += loss.item() * ids.shape[0]; nd += ids.shape[0]
        if i % acc == 0 or i == n_micro:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(trainable, MAX_GRAD_NORM)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        if i >= next_log or i == n_micro:
            next_log += max(1, int(n_micro * log_frac)); print(f"   {tag} {i}/{n_micro} | loss {tot/nd:.4f} | {nd/(time.perf_counter()-t0):.1f} docs/s", flush=True)
    return tot / max(nd, 1), nd / (time.perf_counter() - t0)

@torch.inference_mode()
def predict_logits(model, ds, batch_docs):
    model.eval(); out = []; t0 = time.perf_counter()
    for b in make_loader(ds, batch_docs, False):
        ids, mask, valid = to_dev(b)
        with torch.autocast("cuda", dtype=torch.float16): out.append(model(ids, mask, valid).float().cpu().numpy())
    return np.concatenate(out), len(ds) / (time.perf_counter() - t0)

# ---- auto-tune micro-batch / gradient checkpointing for the partial-tuning model
def autotune_memory():
    global GRAD_CKPT, MB_SEQS
    V = len(RECIPE["views"]); res = []
    for ckpt, seqs in [(False, 16), (False, 8), (True, 16), (True, 8)]:
        model = opt = scaler = None
        try:
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            model = build_model(grad_ckpt=ckpt); model.train(); opt = make_optimizer(model); scaler = torch.amp.GradScaler("cuda"); B = seqs // V
            ids = torch.randint(4, 50_000, (B, V, MAX_LEN), device=device); ids[..., 0] = BOS_ID; ids[..., -1] = EOS_ID
            mask = torch.ones_like(ids, dtype=torch.bool); valid = torch.ones(B, V, dtype=torch.bool, device=device); y = (torch.rand(B, NUM_LABELS, device=device) < 0.03).float()
            for step in range(4):
                if step == 1: torch.cuda.synchronize(); t0 = time.perf_counter()
                with torch.autocast("cuda", dtype=torch.float16): lg = model(ids, mask, valid)
                loss = loss_fn(lg.float(), y); opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            torch.cuda.synchronize(); sps = seqs * 3 / (time.perf_counter() - t0)
            res.append((sps, ckpt, seqs, torch.cuda.max_memory_allocated() / 1024**3)); print(f"grad_ckpt={ckpt!s:5} seqs/micro-batch={seqs:2d}: {sps:5.1f} seq/s | peak {res[-1][3]:.1f} GiB")
        except torch.cuda.OutOfMemoryError:
            print(f"grad_ckpt={ckpt!s:5} seqs/micro-batch={seqs:2d}: OOM")
        finally:
            del model, opt, scaler; gc.collect(); torch.cuda.empty_cache()
    if not res: raise RuntimeError("Nothing fits in GPU memory.")
    sps, GRAD_CKPT, MB_SEQS, _ = max(res); print(f"-> grad_ckpt={GRAD_CKPT}, {MB_SEQS} sequences/micro-batch ({sps:.1f} seq/s)"); return sps

SEQ_PER_S = autotune_memory()
print(f"estimated one full-train epoch: {Ntr * len(RECIPE['views']) / SEQ_PER_S / 60:.0f} min (Exp 1 full fine-tuning: ~57 min incl. dev eval)")

grad_ckpt=False seqs/micro-batch=16:  53.2 seq/s | peak 3.1 GiB
grad_ckpt=False seqs/micro-batch= 8:  49.1 seq/s | peak 2.1 GiB
grad_ckpt=True  seqs/micro-batch=16:  44.7 seq/s | peak 1.4 GiB
grad_ckpt=True  seqs/micro-batch= 8:  41.8 seq/s | peak 1.2 GiB
-> grad_ckpt=False, 16 sequences/micro-batch (53.2 seq/s)
estimated one full-train epoch: 27 min (Exp 1 full fine-tuning: ~57 min incl. dev eval)


## 9. FINAL TRAINING — partial fine-tuning, full train set, 6 epochs, patience 2

In [10]:
FINAL_BEST_PT, FINAL_LAST_PT = RUN_DIR / "final_best.pt", RUN_DIR / "final_last.pt"
FINAL_DEV_LOGITS = RUN_DIR / "final_best_dev_logits.npy"; FINAL_HISTORY = RUN_DIR / "final_history.json"

final_train_idx = np.arange(Ntr) if FINAL_TRAIN_DOCS_LIMIT is None else np.sort(np.random.default_rng(SEED).permutation(Ntr)[:FINAL_TRAIN_DOCS_LIMIT])
final_dev_idx = np.arange(Ndv) if FINAL_TRAIN_DOCS_LIMIT is None else np.arange(min(Ndv, 512))
Y_final_dev = Ydv[final_dev_idx]

seed_everything(SEED)
final_model = build_model()
n_total, n_train = param_report(final_model)
print(f"parameters: total {n_total/1e6:.1f} M | trainable {n_train/1e6:.1f} M ({100*n_train/n_total:.1f} %) | frozen {(n_total-n_train)/1e6:.1f} M")
FT = build_training(final_model, len(final_train_idx), FINAL_MAX_EPOCHS)
final_train_ds = LSIDataset("train", final_train_idx, RECIPE["views"]); final_dev_ds = LSIDataset("dev", final_dev_idx, RECIPE["views"])

history, best_f1, best_epoch, bad, start_epoch = [], -1.0, 0, 0, 1
if RESUME_FINAL and FINAL_LAST_PT.exists():
    ck = torch.load(FINAL_LAST_PT, map_location="cpu", weights_only=False)
    final_model.load_state_dict(ck["model"]); FT["opt"].load_state_dict(ck["opt"]); FT["sched"].load_state_dict(ck["sched"]); FT["scaler"].load_state_dict(ck["scaler"])
    history, best_f1, best_epoch, bad, start_epoch = ck["history"], ck["best_f1"], ck["best_epoch"], ck["bad"], ck["epoch"] + 1
    del ck; print(f"resumed after epoch {start_epoch-1}; best so far {best_f1:.4f} (epoch {best_epoch})")

run_start = time.time()
for epoch in range(start_epoch, FINAL_MAX_EPOCHS + 1):
    ep0 = time.time()
    tr_loss, tr_dps = train_one_epoch(final_model, make_loader(final_train_ds, FT["mb"], True, SEED + epoch), FT, tag=f"partial ep{epoch}/{FINAL_MAX_EPOCHS}")
    z_dev, eval_dps = predict_logits(final_model, final_dev_ds, FT["mb"] * 2); m = evaluate_logits(Y_final_dev, z_dev)
    rec = dict(epoch=epoch, train_loss=tr_loss, train_docs_per_s=tr_dps, **{k: m[k] for k in ["macro_f1_global", "best_global_thr", "macro_ap", "macro_f1_perlabel_cv", "micro_f1", "subset_acc", "macro_precision", "macro_recall", "bce"]}, epoch_minutes=(time.time() - ep0) / 60)
    history.append(rec)
    print(f"\n=== epoch {epoch}: train loss {tr_loss:.4f} | dev macro-F1 {m['macro_f1_global']:.4f} @thr {m['best_global_thr']:.2f} | per-label(CV) {m['macro_f1_perlabel_cv']:.4f} | macro-AP {m['macro_ap']:.4f} | micro-F1 {m['micro_f1']:.4f} | {rec['epoch_minutes']:.1f} min ===\n")
    if m["macro_f1_global"] > best_f1 + 1e-4:
        best_f1, best_epoch, bad = m["macro_f1_global"], epoch, 0
        torch.save({"model_state_dict": {k: v.detach().cpu() for k, v in final_model.state_dict().items()}, "epoch": epoch, "dev_macro_f1_global": best_f1}, FINAL_BEST_PT)
        np.save(FINAL_DEV_LOGITS, z_dev.astype(np.float32)); print(f"   new best epoch {epoch} saved")
    else:
        bad += 1; print(f"   no improvement ({bad}/{FINAL_PATIENCE})")
    json.dump(history, open(FINAL_HISTORY, "w"), indent=1)
    torch.save({"model": final_model.state_dict(), "opt": FT["opt"].state_dict(), "sched": FT["sched"].state_dict(), "scaler": FT["scaler"].state_dict(), "history": history, "best_f1": best_f1, "best_epoch": best_epoch, "bad": bad, "epoch": epoch}, FINAL_LAST_PT)
    if bad >= FINAL_PATIENCE: print("early stopping."); break
train_peak_gib = torch.cuda.max_memory_allocated() / 1024**3
print(f"\nfinished: best epoch {best_epoch}, dev macro-F1 {best_f1:.4f} | wall {(time.time()-run_start)/3600:.2f} h | peak GPU {train_peak_gib:.1f} GiB")
print(f"Exp 1 (full fine-tuning) reference dev macro-F1: {EXP1_DEV_MACRO_F1:.4f}  ->  partial tuning {best_f1:.4f} ({best_f1-EXP1_DEV_MACRO_F1:+.4f})")
display(pd.DataFrame(history).round(4))

parameters: total 139.4 M | trainable 38.0 M (27.3 %) | frozen 101.3 M
   partial ep1/6 534/5344 | loss 0.4848 | 24.6 docs/s
   partial ep1/6 1068/5344 | loss 0.4357 | 24.1 docs/s
   partial ep1/6 1602/5344 | loss 0.4055 | 23.9 docs/s
   partial ep1/6 2136/5344 | loss 0.3876 | 23.9 docs/s
   partial ep1/6 2670/5344 | loss 0.3744 | 23.8 docs/s
   partial ep1/6 3204/5344 | loss 0.3647 | 23.8 docs/s
   partial ep1/6 3738/5344 | loss 0.3571 | 23.7 docs/s
   partial ep1/6 4272/5344 | loss 0.3504 | 23.7 docs/s
   partial ep1/6 4806/5344 | loss 0.3449 | 23.7 docs/s
   partial ep1/6 5340/5344 | loss 0.3402 | 23.7 docs/s
   partial ep1/6 5344/5344 | loss 0.3402 | 23.7 docs/s

=== epoch 1: train loss 0.3402 | dev macro-F1 0.3183 @thr 0.45 | per-label(CV) 0.3342 | macro-AP 0.2908 | micro-F1 0.4193 | 34.1 min ===

   new best epoch 1 saved
   partial ep2/6 534/5344 | loss 0.2709 | 23.6 docs/s
   partial ep2/6 1068/5344 | loss 0.2720 | 23.7 docs/s
   partial ep2/6 1602/5344 | loss 0.2708 | 23.6 doc

,epoch,train_loss,train_docs_per_s,macro_f1_global,best_global_thr,macro_ap,macro_f1_perlabel_cv,micro_f1,subset_acc,macro_precision,macro_recall,bce,epoch_minutes
0,1,0.3402,23.6911,0.3183,0.45,0.2908,0.3342,0.4193,0.0188,0.2901,0.4052,0.1499,34.1076
1,2,0.2646,23.6634,0.3486,0.55,0.3263,0.3624,0.4416,0.0389,0.3393,0.4013,0.1463,34.1492
2,3,0.2063,23.6396,0.3614,0.49,0.3380,0.3727,0.4548,0.0354,0.3331,0.4248,0.1271,34.1801
3,4,0.1371,23.6524,0.3723,0.57,0.3381,0.3755,0.4659,0.0518,0.3656,0.3932,0.1293,34.1677
4,5,0.0866,23.5968,0.3755,0.51,0.3371,0.3767,0.4740,0.0556,0.3815,0.3794,0.1282,34.2605
5,6,0.0581,23.6230,0.3687,0.45,0.3318,0.3707,0.4723,0.0574,0.3877,0.3614,0.1330,34.2230


## 10. Best epoch → post-processing chosen out-of-fold on dev → test predictions → submission + final `.pt`

In [11]:
ck = torch.load(FINAL_BEST_PT, map_location="cpu", weights_only=False)
final_model.load_state_dict(ck["model_state_dict"]); final_model.eval(); best_epoch = ck["epoch"]; del ck
z_dev_final = np.load(FINAL_DEV_LOGITS); assert z_dev_final.shape == Y_final_dev.shape

POST_TABLE, POST_PREDS = strategy_table(Y_final_dev, z_dev_final, K=5)
display(POST_TABLE.round(4))
POST_NAME = POST_TABLE.iloc[0]["strategy"]
post_predictor = STRATEGIES[POST_NAME](Y_final_dev, z_dev_final); dev_oof_pred = POST_PREDS[POST_NAME]
base_row = POST_TABLE[POST_TABLE.strategy == "global"].iloc[0]
print(f"chosen post-processing: {POST_NAME} | OOF dev macro-F1 {POST_TABLE.iloc[0]['macro_f1']:.4f} | micro-F1 {POST_TABLE.iloc[0]['micro_f1']:.4f} | subset acc {POST_TABLE.iloc[0]['subset_acc']:.4f} (plain global: {base_row['macro_f1']:.4f})")

test_ds = LSIDataset("test", np.arange(Nte), RECIPE["views"], with_labels=False)
t0 = time.time(); z_test, test_dps = predict_logits(final_model, test_ds, FT["mb"] * 2); infer_s = time.time() - t0
np.save(RUN_DIR / "test_logits.npy", z_test.astype(np.float32)); print(f"test inference: {Nte:,} docs in {infer_s/60:.1f} min ({test_dps:.1f} docs/s)")
test_pred = post_predictor(z_test)

def write_submission(case_ids, pred_bin, path):
    out = [{"id": cid, "pred": sorted({int(j) for j in np.flatnonzero(pred_bin[i])})} for i, cid in enumerate(case_ids)]
    assert len(out) == len(case_ids) == len(set(map(str, case_ids)))
    for o in out: assert all(isinstance(x, int) and 0 <= x < NUM_LABELS for x in o["pred"]) and len(o["pred"]) == len(set(o["pred"]))
    json.dump(out, open(path, "w")); return out

test_ids = ARR["test"]["ids"]
submission = write_submission(test_ids, test_pred, RUN_DIR / "submission.json")
if ON_KAGGLE: write_submission(test_ids, test_pred, WORK / "submission_exp2_partial.json")
n_lab = np.array([len(o["pred"]) for o in submission])
print(f"submission: {len(submission):,} objects | labels/doc mean {n_lab.mean():.2f}, min {n_lab.min()}, max {n_lab.max()} | empty {int((n_lab==0).sum())}")

FINAL_PT = RUN_DIR / "lsi_encoder_partial_ft_final.pt"
post = dict(name=POST_NAME, thresholds=np.asarray(post_predictor.thr).tolist(), calibrator=post_predictor.tf.params, ensure_at_least_one_label=True)
best_state = torch.load(FINAL_BEST_PT, map_location="cpu", weights_only=False)["model_state_dict"]
torch.save({"model_state_dict": best_state, "recipe": {**RECIPE, "views": list(RECIPE["views"])}, "partial_tuning": {"unfreeze_top_n": UNFREEZE_TOP_N, "n_frozen": N_FROZEN, "embeddings_frozen": True},
            "encoder_config": asdict(cfg), "num_labels": NUM_LABELS, "best_epoch": best_epoch, "dev_metrics_best_epoch": history[best_epoch - 1] if history else None, "postprocess": post,
            "dev_oof_macro_f1_after_postprocess": float(POST_TABLE.iloc[0]["macro_f1"]), "max_seq_len": MAX_LEN, "content_tokens_per_view": CONTENT,
            "tokenizer_path": str(PHASE1_TOKENIZER_PATH), "phase1_checkpoint": str(PHASE1_CHECKPOINT_PATH), "experiment_2": "phase1 encoder + partial fine-tuning"}, FINAL_PT)
del best_state; gc.collect()
print(f"saved {FINAL_PT} ({FINAL_PT.stat().st_size/1024**2:.0f} MB)")

,strategy,macro_f1,micro_f1,macro_precision,macro_recall,subset_acc,avg_pred,empty_rate,seconds
0,per-label shrunk(100) + top1,0.3764,0.4749,0.3779,0.3872,0.0564,4.6268,0.0000,2.0466
1,per-label shrunk(300) + top1,0.3761,0.4757,0.3760,0.3869,0.0543,4.5543,0.0000,1.9504
2,per-label raw + top1,0.3758,0.4713,0.3852,0.3881,0.0537,4.7315,0.0000,1.8288
3,temperature -> global,0.3748,0.4735,0.3797,0.3798,0.0571,4.1609,0.0000,2.9907
4,global,0.3747,0.4739,0.3776,0.3816,0.0544,4.2017,0.0491,0.9604
5,global + top1,0.3741,0.4735,0.3749,0.3832,0.0554,4.2509,0.0000,0.9124
6,Platt -> per-label shrunk(100),0.3722,0.4790,0.3932,0.3748,0.0548,4.6169,0.0000,4.1858
7,stacker -> per-label shrunk(100),0.3636,0.4757,0.3673,0.3716,0.0503,4.6363,0.0000,31.0449
8,Platt -> global,0.3619,0.4769,0.3794,0.3887,0.0359,5.6036,0.0000,3.2808
9,stacker -> global,0.3591,0.4811,0.3569,0.3761,0.0465,4.9185,0.0000,30.3581


chosen post-processing: per-label shrunk(100) + top1 | OOF dev macro-F1 0.3764 | micro-F1 0.4749 | subset acc 0.0564 (plain global: 0.3747)
test inference: 13,019 docs in 5.1 min (42.2 docs/s)
submission: 13,019 objects | labels/doc mean 4.64, min 1, max 26 | empty 0
saved /kaggle/working/lsi_exp2_partial/lsi_encoder_partial_ft_final.pt (532 MB)


## 11. Analysis for the report

In [12]:
y, pr = Y_final_dev, dev_oof_pred
f1 = label_f1(y, pr); sup_tr = Ytr.sum(0); sup_dv = y.sum(0); yb, pb = y.astype(bool), pr.astype(bool)
tp = (yb & pb).sum(0); fp = (~yb & pb).sum(0); fn = (yb & ~pb).sum(0)
per_label = pd.DataFrame({"statute": np.arange(NUM_LABELS), "train_support": sup_tr, "dev_support": sup_dv, "precision": tp / np.maximum(tp + fp, 1), "recall": tp / np.maximum(tp + fn, 1), "f1": f1, "FP": fp, "FN": fn})
per_label.to_csv(RUN_DIR / "partial_per_label_metrics.csv", index=False)
per_label["support_quartile"] = pd.qcut(per_label["train_support"].rank(method="first"), 4, labels=["rarest 25%", "rare-mid", "common-mid", "most common 25%"])
print("F1 by statute frequency:"); display(per_label.groupby("support_quartile", observed=True)[["f1", "precision", "recall"]].mean().round(3))
print("worst statutes:"); display(per_label.sort_values("f1").head(10).round(3).reset_index(drop=True))
print("best statutes:"); display(per_label.sort_values("f1", ascending=False).head(8).round(3).reset_index(drop=True))
print("corr(log train support, F1) = %.3f" % np.corrcoef(np.log(sup_tr), f1)[0, 1])

M = (yb & ~pb).T.astype(np.int64) @ (~yb & pb).astype(np.int64); np.fill_diagonal(M, 0)
top = np.dstack(np.unravel_index(np.argsort(-M.ravel())[:10], M.shape))[0]
print("\nmost common 'missed j / predicted-instead k' pairs:")
for j, k in top: print(f"  missed {j:>2d} / predicted {k:>2d}: {M[j,k]:4d} docs")

# per-label F1 comparison with Exp 1 if its file is available
exp1_csv = WORK / "lsi_runs" / "final_per_label_metrics.csv"
if exp1_csv.exists():
    e1 = pd.read_csv(exp1_csv); print(f"\nper-label F1: partial {per_label['f1'].mean():.4f} vs full-FT {e1['f1'].mean():.4f} | partial better on {(per_label['f1'].values > e1['f1'].values).sum()} / {NUM_LABELS} statutes")

print("\n================ COST SUMMARY (Exp 2: partial fine-tuning) ================")
print(f"trainable params: {n_train/1e6:.1f} M of {n_total/1e6:.1f} M ({100*n_train/n_total:.1f} %) | inference params: {n_total/1e6:.1f} M (unchanged)")
print(f"train throughput: {np.mean([h['train_docs_per_s'] for h in history]):.1f} docs/s | epoch time {np.mean([h['epoch_minutes'] for h in history]):.1f} min (Exp 1: ~13.5 docs/s, 56.8 min) | peak train GPU {train_peak_gib:.1f} GiB (Exp 1 exp-runs: 7.7 GiB)")
print(f"test inference: {test_dps:.1f} docs/s | grad checkpointing {GRAD_CKPT} | micro-batch seqs {MB_SEQS}")
print(f"best epoch {best_epoch} | dev macro-F1 (global thr) {best_f1:.4f} | after post-processing (OOF) {POST_TABLE.iloc[0]['macro_f1']:.4f} | Exp 1 reference {EXP1_DEV_MACRO_F1:.4f}")

F1 by statute frequency:


,f1,precision,recall
support_quartile,,,
rarest 25%,0.269,0.298,0.256
rare-mid,0.354,0.373,0.344
common-mid,0.389,0.382,0.406
most common 25%,0.493,0.459,0.543


worst statutes:


,statute,train_support,dev_support,precision,recall,f1,FP,FN,support_quartile
0,8,349,92,0.098,0.065,0.078,55,86,rarest 25%
1,65,351,105,0.159,0.105,0.126,58,94,rarest 25%
2,98,490,123,0.114,0.154,0.131,148,104,rare-mid
3,0,343,86,0.189,0.116,0.144,43,76,rarest 25%
4,52,1084,272,0.141,0.158,0.149,263,229,common-mid
5,40,543,152,0.126,0.217,0.159,229,119,rare-mid
6,2,367,98,0.235,0.122,0.161,39,86,rarest 25%
7,32,385,99,0.172,0.152,0.161,72,84,rarest 25%
8,76,530,142,0.159,0.169,0.164,127,118,rare-mid
9,99,674,176,0.198,0.142,0.166,101,151,rare-mid


best statutes:


,statute,train_support,dev_support,precision,recall,f1,FP,FN,support_quartile
0,28,602,151,0.840,0.834,0.837,24,25,rare-mid
1,34,10380,2596,0.762,0.800,0.780,648,520,most common 25%
2,20,487,122,0.736,0.779,0.757,34,27,rare-mid
3,81,1321,346,0.755,0.740,0.747,83,90,common-mid
4,94,3835,958,0.739,0.661,0.698,224,325,most common 25%
5,29,497,125,0.758,0.600,0.670,24,50,rare-mid
6,37,1269,318,0.625,0.654,0.639,125,110,common-mid
7,61,3723,932,0.621,0.611,0.616,347,363,most common 25%


corr(log train support, F1) = 0.536

most common 'missed j / predicted-instead k' pairs:
  missed 13 / predicted  5:  195 docs
  missed 39 / predicted 43:  189 docs
  missed 12 / predicted  5:  188 docs
  missed 39 / predicted  5:  171 docs
  missed 14 / predicted  5:  154 docs
  missed 39 / predicted 97:  154 docs
  missed 97 / predicted  5:  134 docs
  missed 13 / predicted 43:  131 docs
  missed 10 / predicted  5:  126 docs
  missed 34 / predicted  5:  122 docs

================ COST SUMMARY (Exp 2: partial fine-tuning) ================
trainable params: 38.0 M of 139.4 M (27.3 %) | inference params: 139.4 M (unchanged)
train throughput: 23.6 docs/s | epoch time 34.2 min (Exp 1: ~13.5 docs/s, 56.8 min) | peak train GPU 3.1 GiB (Exp 1 exp-runs: 7.7 GiB)
test inference: 42.2 docs/s | grad checkpointing False | micro-batch seqs 16
best epoch 5 | dev macro-F1 (global thr) 0.3755 | after post-processing (OOF) 0.3764 | Exp 1 reference 0.3826
